In [3]:
# Import required libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# 1. Create a sample dataset
# age and salary are input features
# target is the output we want to predict

data = pd.DataFrame({
    "age": [20, 25, 30, 35, 40, 45, 50, 55],
    "salary": [20000, 25000, 30000, 35000, 40000, 45000, 50000, 55000],
    "target": [0, 0, 0, 1, 1, 1, 1, 1]
})

print("Dataset:")
print(data)


# 2. Separate input features (X) and target (y)

X = data[["age", "salary"]]
y = data["target"]


# 3. Split the dataset into training and testing data
# 80% → training
# 20% → testing

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# 4. Create the Random Forest classification model

model = RandomForestClassifier(random_state=42)


# 5. Train the model using training data

model.fit(X_train, y_train)


# 6. Make predictions using test data

y_pred = model.predict(X_test)


# 7. Calculate the accuracy of the model

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", accuracy)

Dataset:
   age  salary  target
0   20   20000       0
1   25   25000       0
2   30   30000       0
3   35   35000       1
4   40   40000       1
5   45   45000       1
6   50   50000       1
7   55   55000       1

Model Accuracy: 1.0


In [2]:
# ================================================================
# Experiment 3: End-to-End Automated Data Pipeline (ETL + Train)
# ================================================================
# Stages:
# Extract -> Validate -> Transform -> Train -> Evaluate -> Persist
# ================================================================

import os
import joblib
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


class DataPipeline:
    """A minimal but complete automated ML data pipeline."""

    def __init__(self, artifact_dir="artifacts"):
        self.artifact_dir = artifact_dir
        self.scaler = StandardScaler()
        self.model = RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )

        # Create artifact directory if it does not exist
        os.makedirs(artifact_dir, exist_ok=True)

    # ------------------------------------------------------------
    # Stage 1: Extract
    # ------------------------------------------------------------
    def extract(self):
        data = load_iris(as_frame=True)
        df = data.frame

        print(
            f"[EXTRACT] loaded {df.shape[0]} rows x "
            f"{df.shape[1]} cols"
        )

        return df

    # ------------------------------------------------------------
    # Stage 2: Validate
    # ------------------------------------------------------------
    def validate(self, df):
        missing = int(df.isnull().sum().sum())

        assert missing == 0, "Data contains missing values!"
        assert "target" in df.columns, "Target column missing!"

        print(
            "[VALIDATE] no missing values, "
            "target present -> OK"
        )

        return df

    # ------------------------------------------------------------
    # Stage 3: Transform
    # ------------------------------------------------------------
    def transform(self, df):
        X = df.drop(columns=["target"])
        y = df["target"]

        X_scaled = self.scaler.fit_transform(X)

        print(
            f"[TRANSFORM] standard-scaled "
            f"{X.shape[1]} features"
        )

        return X_scaled, y

    # ------------------------------------------------------------
    # Stage 4 + 5: Train and Evaluate
    # ------------------------------------------------------------
    def train(self, X, y):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
            stratify=y
        )

        # Train model
        self.model.fit(X_tr, y_tr)

        # Make predictions
        y_pred = self.model.predict(X_te)

        # Calculate accuracy
        acc = accuracy_score(y_te, y_pred)

        print(
            f"[TRAIN] model fitted, "
            f"test accuracy = {acc:.3f}"
        )

        # Classification report
        print("[REPORT] per-class metrics:")
        print(
            classification_report(
                y_te,
                y_pred,
                digits=2
            )
        )

        return acc

    # ------------------------------------------------------------
    # Stage 6: Persist
    # ------------------------------------------------------------
    def persist(self):
        model_path = os.path.join(
            self.artifact_dir,
            "model.pkl"
        )

        scaler_path = os.path.join(
            self.artifact_dir,
            "scaler.pkl"
        )

        # Save model and scaler
        joblib.dump(self.model, model_path)
        joblib.dump(self.scaler, scaler_path)

        print(
            f"[PERSIST] saved model -> {model_path}"
        )

        print(
            f"[PERSIST] saved scaler -> {scaler_path}"
        )

    # ------------------------------------------------------------
    # Orchestration
    # ------------------------------------------------------------
    def run(self):
        print("=== Automated Data Pipeline Started ===")

        # Extract
        df = self.extract()

        # Validate
        df = self.validate(df)

        # Transform
        X, y = self.transform(df)

        # Train + Evaluate
        self.train(X, y)

        # Persist
        self.persist()

        print("=== Pipeline Completed Successfully ===")


# ------------------------------------------------------------
# Main Program
# ------------------------------------------------------------
if __name__ == "__main__":
    pipeline = DataPipeline()
    pipeline.run()

=== Automated Data Pipeline Started ===
[EXTRACT] loaded 150 rows x 5 cols
[VALIDATE] no missing values, target present -> OK
[TRANSFORM] standard-scaled 4 features
[TRAIN] model fitted, test accuracy = 0.933
[REPORT] per-class metrics:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

[PERSIST] saved model -> artifacts/model.pkl
[PERSIST] saved scaler -> artifacts/scaler.pkl
=== Pipeline Completed Successfully ===
